In [ ]:
from pypdf import PdfReader

# 1. 打开这本书
reader = PdfReader("my_doc.pdf")

# 2. 看看这本书一共多少页
print(f"这本书一共有 {len(reader.pages)} 页。")

# 3. 翻到第一页，把上面的字全部抠下来
page_one = reader.pages[0]
text = page_one.extract_text()

# 4. 把抠下来的字打印出来看看
print(text)

In [ ]:
class TextSplitter:
    def __init__(self,chunk_size=100,chunk_overlap=20):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    def split (self,text):
        chunks = []
        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunks.append(text[start:end])
            start = end - self.chunk_overlap
        return chunks
splitter = TextSplitter()
chunks = splitter.split(text)
print(f"这一页被切成了 {len(chunks)} 块。\n")
for i in range(3):
    print(f"--- 第 {i+1} 块 ---")
    print(chunks[i])
    print()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. 把切好的文字块放进 TF-IDF 引擎（这是业界常用的文本向量化工具）
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(chunks)

# 2. 模拟用户提问
user_question = "代表性传承人级别是什么？"
print(f"🙋 用户提问: {user_question}\n")

# 3. 把用户的问题也变成向量
question_vector = vectorizer.transform([user_question])

# 4. 计算用户问题和所有豆腐块的“相似度（夹角余弦）”
similarities = cosine_similarity(question_vector, tfidf_matrix)

# 5. 找出得分最高的那一块的索引
best_match_index = similarities.argmax()

# 6. 把最好的那一块打印出来
print("🎯 系统检索到的最相关文本如下：\n")
print(chunks[best_match_index])


In [ ]:
import requests

# 1. 这是你刚才找出来的那块小饼干
best_chunk = chunks[best_match_index]

# 2. 拼装终极 Prompt（这就是 RAG 的“生成”环节）
prompt = f"""
请根据以下参考文档，用最简练的语言回答用户的问题。如果文档中没有答案，请直接说不知道。
【参考文档】：{best_chunk}

【用户问题】：{user_question}
"""

# 3. 调用你的 DeepSeek API（去把你昨天写的那个函数搬过来，或者直接写最基础的请求）
api_key = ""
url = "https://api.deepseek.com/chat/completions"
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
data = {
    "model": "deepseek-chat",
    "messages": [{"role": "user", "content": prompt}]
}

response = requests.post(url, headers=headers, data=json.dumps(data, ensure_ascii=False).encode('utf-8'))
print("🤖 大模型基于你的PDF给出的回答：")
print(response.json()["choices"][0]["message"]["content"])

In [ ]:
from pypdf import PdfReader

reader = PdfReader("my_doc.pdf")
full_text = ""

# 遍历这本书的每一页，把所有字拼起来
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        full_text += page_text + "\n"

print(f"✅ 成功提取全本文字，共计 {len(full_text)} 个字。")

# 然后用咱们的切块机，把它切成小块
splitter = TextSplitter(chunk_size=300, chunk_overlap=50) # 这里我把块调大了一点
chunks = splitter.split(full_text)
print(f"🍰 全书被切成了 {len(chunks)} 块。")

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_deepseek import ChatDeepSeek

# 1. 填入你的 DeepSeek API Key（记得别把真实的传上 GitHub）
os.environ["DEEPSEEK_API_KEY"] = ""

# 2. 加载 PDF（LangChain 自带终极解析器）
loader = PyPDFLoader("my_doc.pdf")
docs = loader.load()

# 3. 智能切块（按段落、标点切，绝不把一句话劈两半）
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, 
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "！", "？", " ", ""] # 智能优先级
)
splits = text_splitter.split_documents(docs)
print(f"🍰 LangChain 智能切块完成，共切出 {len(splits)} 块。")

# 4. 初始化 DeepSeek 大模型
llm = ChatDeepSeek(model="deepseek-chat", temperature=0)

# 5. 让大模型直接读取前 5 块内容并回答问题（极简演示版）
print("\n🤖 大模型正在基于切块内容作答...")
question = "代表性传承人级别是什么？"

# 拼装上下文（这里我们为了演示，直接把所有切块拼接，真实企业会放到向量数据库）
context = "\n\n".join([doc.page_content for doc in splits[:5]]) 

prompt = f"""
请仅根据以下提供的上下文，用最简练的语言回答用户的问题。如果上下文没有提到，请直接说“不知道”。
【上下文】：{context}
【用户问题】：{question}
"""

response = llm.invoke(prompt)
print(f"\n✨ 最终回答：{response.content}")
try:
    # 如果一切正常
    response = llm.invoke(prompt)
    print(f"\n✨ 最终回答：{response.content}")
except Exception as e:
    # 如果网断了、API挂了，系统不会崩溃，而是给用户一个友好的提示
    print(f"\n⚠️ 系统提示：AI 服务暂时不可用，请稍后再试。（错误码：{e}）")

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_deepseek import ChatDeepSeek
# 🌟 引入真正的本地向量模型
from langchain_huggingface import HuggingFaceEmbeddings

class AdvancedKnowledgeTool:
    def __init__(self, pdf_path, api_key):
        self.name = "search_knowledge"
        self.description = "当你需要查询长篇文档（如PDF）中的具体信息时，请调用此工具，传入你要查的问题。"
        
        print("⏳ 正在初始化 RAG 引擎（真实向量版）...")
        self.llm = ChatDeepSeek(model="deepseek-chat", api_key=api_key, temperature=0)
        
        # 🌟 下载并加载真实的向量模型（all-MiniLM-L6-v2 是业界极轻量极好用的模型）
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
        self.splits = text_splitter.split_documents(docs)
        
        # 用真实的 embeddings 来建库
        self.vectorstore = Chroma.from_documents(self.splits, self.embeddings)
        print(f"✅ RAG 引擎初始化完毕！共载入 {len(self.splits)} 个文本块。")

    def execute(self, args):
        query = args.get("query", "")
        if not query:
            return "未提供查询内容。"
            
        # 真正去向量库里搜索
        docs = self.vectorstore.similarity_search(query, k=3)
        context = "\n\n".join([doc.page_content for doc in docs])
        
        prompt = f"""
        请仅根据以下提供的上下文，用最简练的语言回答用户的问题。如果上下文没有提到，请直接说“不知道”。
        【上下文】：{context}
        【用户问题】：{query}
        """
        try:
            response = self.llm.invoke(prompt)
            return response.content
        except Exception as e:
            return f"查询失败：{e}"

# ==========================================
# 测试真正的向量检索
# ==========================================
my_key = ""
rag_tool = AdvancedKnowledgeTool("my_doc.pdf", my_key) 
print("--- 测试查询 ---")
print(rag_tool.execute({"query": "代表性传承人级别是什么？"}))

C:\Users\Administrator\AppData\Local\Temp\ipykernel_41656\2205577624.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


⏳ 正在初始化 RAG 引擎（真实向量版）...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ RAG 引擎初始化完毕！共载入 16 个文本块。
--- 测试查询 ---
县级
